# Evaluate GRPO Checkpoint 25 from the Downloaded Archive

Select a **T4 GPU**, run the cell, and upload `code-grpo-humaneval-50-step-improved-results.zip` when prompted. The training is not repeated.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import files

REPOSITORY = 'https://github.com/Hamza-Nadif/code-grpo-humaneval.git'
WORKDIR = Path('/content/code-grpo-humaneval-checkpoint-eval')
ARTIFACTS = Path('/content/grpo-50-step-artifacts')
RESULTS = Path('/content/checkpoint-25-evaluation')

def run(command):
    print('\n$', ' '.join(map(str, command)), flush=True)
    subprocess.run([str(part) for part in command], check=True)

print('STEP 1/5 - Checking the GPU', flush=True)
run(['nvidia-smi'])

print('STEP 2/5 - Uploading the completed 50-step archive', flush=True)
uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError('Upload exactly one ZIP archive.')
archive = Path('/content') / next(iter(uploaded))
if archive.suffix.lower() != '.zip':
    raise RuntimeError('The uploaded file must be a ZIP archive.')
if ARTIFACTS.exists():
    shutil.rmtree(ARTIFACTS)
ARTIFACTS.mkdir(parents=True)
shutil.unpack_archive(archive, ARTIFACTS)
adapter = ARTIFACTS / 'adapter' / 'checkpoint-25'
if not (adapter / 'adapter_model.safetensors').exists():
    raise RuntimeError('Checkpoint 25 was not found in the uploaded archive.')

print('STEP 3/5 - Loading the evaluation code', flush=True)
if WORKDIR.exists():
    run(['git', '-C', WORKDIR, 'pull', '--ff-only', 'origin', 'main'])
else:
    run(['git', 'clone', '--branch', 'main', REPOSITORY, WORKDIR])
os.chdir(WORKDIR)
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
run([sys.executable, 'build_training_data.py', '--output-dir', 'data'])

print('STEP 4/5 - Evaluating checkpoint 25 without retraining', flush=True)
run([
    sys.executable, 'evaluate_baseline.py',
    '--data', 'data/humaneval_test.jsonl',
    '--backend', 'transformers',
    '--model', 'Qwen/Qwen2.5-Coder-0.5B-Instruct',
    '--adapter', adapter,
    '--quantization', '4bit',
    '--samples-per-task', '1',
    '--max-new-tokens', '256',
    '--temperature', '0',
    '--executor', 'local',
    '--allow-local-code-execution',
    '--output-dir', RESULTS,
])

print('STEP 5/5 - Reporting and downloading the result', flush=True)
summary = json.loads((RESULTS / 'summary.json').read_text())
score = summary['metrics']['pass@1']
print(f'Checkpoint 25 pass@1: {score:.4f}')
print(f'Baseline to beat:       {0.4091:.4f}')
print(f'Difference:             {score - 0.4091:+.4f}')
result_archive = shutil.make_archive('/content/checkpoint-25-evaluation', 'zip', RESULTS)
files.download(result_archive)
